In [ ]:
from pathlib import Path
import sys
import os

import torch
import numpy as np
import open3d as o3d

from video_depth_anything.video_depth import VideoDepthAnything
from utils.dc_utils import read_video_frames

sys.path.append(str(Path.cwd().parent))
from src.common.visualize.pointcloud import plot_pointcloud

ROOT = Path.cwd().parent / "Video-Depth-Anything"
ENCODER_NAME = "vitl" # encoder name, can be "vits", "vitb" or "vitl"
METRIC = True

INPUT_SIZE = 518 # input size for the model
FP32 = True # If True, the model will run in fp32 mode, otherwise it will run in fp16 mode. Note that fp16 mode is faster but less accurate.

# Parameters for frame extraction, these are not used if the input is a folder of images
MAX_LEN = -1 # maximum length of the input video, -1 means no limit
TARGET_FPS = -1 # target fps of the input video, -1 means the original fps
MAX_RES = 1280 # maximum resolution of the input video

# Parameters for post processing, these are not used for the model itself
FOCAL_LENGTH_X = 470.4 # Focal length along the x-axis
FOCAL_LENGTH_Y = 470.4 # Focal length along the y-axis

INPUT_VIDEO = ROOT / "assets/example_videos/davis_rollercoaster.mp4"
OUTPUT_DIR = ROOT / "outputs"

device = "cuda" if torch.cuda.is_available() else "cpu"

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
}
checkpoint_name = 'metric_video_depth_anything' if METRIC else 'video_depth_anything'

video_depth_anything = VideoDepthAnything(**model_configs[ENCODER_NAME], metric=METRIC)
video_depth_anything.load_state_dict(torch.load(f'{ROOT}/checkpoints/{checkpoint_name}_{ENCODER_NAME}.pth', map_location='cpu'), strict=True)
video_depth_anything = video_depth_anything.to(device).eval()

frames, target_fps = read_video_frames(INPUT_VIDEO, MAX_LEN, TARGET_FPS, MAX_RES) # frames: uint8 ndarray shape (N, H, W, 3), target_fps: float
depths, fps = video_depth_anything.infer_video_depth(frames, target_fps,  # depths: float32 ndarray shape (N, H, W), fps: float
                                                     input_size=INPUT_SIZE, device=device, fp32=FP32)

if METRIC:
    width, height = depths[0].shape[-1], depths[0].shape[-2]
    x, y = np.meshgrid(np.arange(width), np.arange(height))
    x = (x - width / 2) / FOCAL_LENGTH_X
    y = (y - height / 2) / FOCAL_LENGTH_Y

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for i, (color_image, depth) in enumerate(zip(frames, depths)):
        z = np.array(depth)
        points = np.stack((np.multiply(x, z), np.multiply(y, z), z), axis=-1).reshape(-1, 3)
        colors = np.array(color_image).reshape(-1, 3) / 255.0

        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points)
        pcd.colors = o3d.utility.Vector3dVector(colors)
        o3d.io.write_point_cloud(os.path.join(OUTPUT_DIR, 'point' + str(i).zfill(4) + '.ply'), pcd)
        print(f"Saved point cloud {i} to {os.path.join(OUTPUT_DIR, 'point' + str(i).zfill(4) + '.ply')}")

In [ ]:
# Plot the first frame result
import plotly.graph_objects as go
from open3d.visualization.draw_plotly import get_plotly_fig

if METRIC:
    color_image, depth = frames[0], depths[0]
    width, height = depth.shape[-1], depth.shape[-2]

    ### Depth image visualization ###
    fig = go.Figure()
    fig.add_trace(
        go.Heatmap(
            z=depth,
            x=np.arange(width),
            y=np.arange(height),
            colorscale="Turbo",
            hovertemplate=(
                "x: %{x}<br>"
                "y: %{y}<br>"
                f"depth: %{{z:.3f}} m"
                "<extra></extra>"
            ),
            colorbar=dict(title="Depth (m)"),
        )
    )
    fig.update_xaxes(constrain="domain")
    fig.update_yaxes(autorange="reversed",
                        scaleanchor="x",
                        scaleratio=1)
    fig.update_layout(
        xaxis_title="x [pixel]",
        yaxis_title="y [pixel]",
    )
    fig.show()

    ### Point cloud visualization ###
    x, y = np.meshgrid(np.arange(width), np.arange(height))
    x = (x - width / 2) / FOCAL_LENGTH_X
    y = (y - height / 2) / FOCAL_LENGTH_Y
    z = np.array(depth)
    points = np.stack((np.multiply(x, z), np.multiply(y, z), z), axis=-1).reshape(-1, 3)
    colors = np.array(color_image).reshape(-1, 3) / 255.0
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    # Downsample is needed because the number of points is too large for Plotly to handle, otherwise it will crash the browser
    downpcd = pcd.voxel_down_sample(voxel_size=5.0)

    # Create a Plotly figure from the Open3D PointCloud
    fig = get_plotly_fig(
        [downpcd],
        width=640,
        height=480,
    )
    fig.update_traces(
        marker=dict(
            size=0.5,
            opacity=0.4,
        ),
        selector=dict(type="scatter3d"),
    )
    fig.show()